# MechaCare Industrial Systems — Machine Maintenance Insight

## Problem Statement and Objectives
This notebook audits the machine maintenance dataset, cleans it, validates joins, performs exploratory analysis, and answers six business questions using evidence from the extracted CSVs. The raw files are preserved and never overwritten.

**Assessment use case:** 25 — Machine Maintenance Insight
**Scope:** sensor, machine, usage, alarm, and maintenance data


In [21]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'raw'
CLEAN_DIR = BASE_DIR / 'cleaned'
OUT_DIR = BASE_DIR / 'outputs'
PLOTS_DIR = BASE_DIR / 'plots'

for d in [CLEAN_DIR, OUT_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', BASE_DIR)
print('Raw files:', sorted(p.name for p in RAW_DIR.glob('*.csv')))


Project root: c:\Users\Hari Vignesh S\Downloads\MechaCare_Final_Assessment_Package1\MechaCare_Final_Assessment_Package
Raw files: ['alarms.csv', 'machines.csv', 'maintenance.csv', 'sensor.csv', 'usage.csv']


In [22]:
sensor = pd.read_csv(RAW_DIR / 'sensor.csv')
machines = pd.read_csv(RAW_DIR / 'machines.csv')
usage = pd.read_csv(RAW_DIR / 'usage.csv')
alarms = pd.read_csv(RAW_DIR / 'alarms.csv')
maintenance = pd.read_csv(RAW_DIR / 'maintenance.csv')

datasets = {
    'sensor': sensor,
    'machines': machines,
    'usage': usage,
    'alarms': alarms,
    'maintenance': maintenance,
}

for name, df in datasets.items():
    print(f'\n{name.upper()}')
    print('shape=', df.shape)
    print('columns=', list(df.columns))
    print('dtypes=')
    print(df.dtypes)
    print('missing_values=')
    print(df.isnull().sum())
    print('duplicate_rows=', int(df.duplicated().sum()))
    print(df.head(3).to_string(index=False))



SENSOR
shape= (31011, 5)
columns= ['timestamp', 'machine', 'vibration', 'temperature', 'current']
dtypes=
timestamp       object
machine         object
vibration      float64
temperature    float64
current        float64
dtype: object
missing_values=
timestamp       0
machine         0
vibration      12
temperature    12
current        11
dtype: int64
duplicate_rows= 1
          timestamp machine  vibration  temperature  current
2026-01-01 00:00:00    M001       2.16         76.1    12.04
2026-01-01 12:00:00    M001       3.12         75.1    13.49
2026-01-02 00:00:00    M001       2.22         74.4    11.18

MACHINES
shape= (60, 4)
columns= ['machine', 'type', 'age', 'installation_date']
dtypes=
machine               object
type                  object
age                  float64
installation_date     object
dtype: object
missing_values=
machine              0
type                 0
age                  0
installation_date    0
dtype: int64
duplicate_rows= 0
machine            type 

## Initial Data Inspection
The raw tables have different grains: sensor readings are repeated over time, usage is weekly by machine, alarms are event-level, and maintenance is event-level. Before combining any tables, we confirm the natural key and the row definitions.


In [23]:
# Inspect duplicate business keys
print('Sensor duplicate (machine, timestamp):', sensor.duplicated(subset=['machine', 'timestamp']).sum())
print('Alarm duplicate (machine, timestamp, alarm_type):', alarms.duplicated(subset=['machine', 'timestamp', 'alarm_type']).sum())
print('Sensor duplicate machine/timestamp examples:')
print(sensor[sensor.duplicated(subset=['machine', 'timestamp'], keep=False)].head(10).to_string(index=False))
print('\nAlarm label counts before normalization:')
print(alarms['alarm_type'].value_counts(dropna=False).head(20).to_string())

# Convert datetime columns
sensor['timestamp'] = pd.to_datetime(sensor['timestamp'], errors='coerce')
machines['installation_date'] = pd.to_datetime(machines['installation_date'], errors='coerce')
usage['week_start'] = pd.to_datetime(usage['week_start'], errors='coerce')
alarms['timestamp'] = pd.to_datetime(alarms['timestamp'], errors='coerce')
maintenance['service_date'] = pd.to_datetime(maintenance['service_date'], errors='coerce')

print('\nInvalid timestamps:')
for name, df, col in [('sensor', sensor, 'timestamp'), ('alarms', alarms, 'timestamp'), ('machines', machines, 'installation_date'), ('maintenance', maintenance, 'service_date')]:
    if col in df.columns:
        print(f'{name}.{col}:', df[col].isna().sum())


Sensor duplicate (machine, timestamp): 1
Alarm duplicate (machine, timestamp, alarm_type): 1
Sensor duplicate machine/timestamp examples:
          timestamp machine  vibration  temperature  current
2026-01-18 00:00:00    M059       3.51         61.6    10.58
2026-01-18 00:00:00    M059       3.51         61.6    10.58

Alarm label counts before normalization:
alarm_type
Warning             219
warning             213
general_warning     205
High Vibration       83
 high vibration      68
high_vibration       67
high_temperature     23
temp_high            22
High Temperature     19
over current         13
overcurrent          11
NaN                   4
Over Current          2

Invalid timestamps:
sensor.timestamp: 0
alarms.timestamp: 0
machines.installation_date: 0
maintenance.service_date: 0


## Data Quality Assessment

This section deliberately distinguishes between:
- exact duplicate records
- duplicate business keys where repeated observations are legitimate across machines
- missing values and domain values that require context
- statistically unusual but possibly valid sensor readings


In [ ]:
# Build a data quality summary
quality_rows = []
for name, df in datasets.items():
    row = {
        'table': name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_values_total': int(df.isna().sum().sum()),
    }
    if 'machine' in df.columns and 'timestamp' in df.columns:
        row['duplicate_machine_timestamp_keys'] = int(df.duplicated(subset=['machine', 'timestamp']).sum())
    else:
        row['duplicate_machine_timestamp_keys'] = np.nan
    quality_rows.append(row)

quality_summary = pd.DataFrame(quality_rows)
print(quality_summary.to_string(index=False))
quality_summary.to_csv(OUT_DIR / 'data_quality_report.csv', index=False)


In [24]:
# Clean a copy of each table without mutating the raw files
sensor_clean = sensor.copy()
machines_clean = machines.copy()
usage_clean = usage.copy()
alarms_clean = alarms.copy()
maintenance_clean = maintenance.copy()

for df in [sensor_clean, machines_clean, usage_clean, alarms_clean, maintenance_clean]:
    df.columns = df.columns.str.strip()

# Remove exact duplicates where justified
sensor_clean = sensor_clean.drop_duplicates().copy()
usage_clean = usage_clean.drop_duplicates().copy()
alarms_clean = alarms_clean.drop_duplicates().copy()
maintenance_clean = maintenance_clean.drop_duplicates().copy()
machines_clean = machines_clean.drop_duplicates().copy()

# Handle sensor timestamps
sensor_clean['timestamp'] = pd.to_datetime(sensor_clean['timestamp'], errors='coerce')
sensor_clean = sensor_clean.sort_values(['machine', 'timestamp']).reset_index(drop=True)
sensor_clean = sensor_clean.drop_duplicates(subset=['machine', 'timestamp'], keep='first').reset_index(drop=True)

# Preserve missingness information instead of zero-filling
sensor_clean['vibration_missing'] = sensor_clean['vibration'].isna()
sensor_clean['temperature_missing'] = sensor_clean['temperature'].isna()
sensor_clean['current_missing'] = sensor_clean['current'].isna()

# Standardize alarm labels
def normalize_alarm(value):
    if pd.isna(value):
        return 'unknown'
    v = str(value).strip().lower()
    v = re.sub(r'\s+', ' ', v)
    mapping = {
        'warning': 'general_warning',
        'general_warning': 'general_warning',
        'high vibration': 'high_vibration',
        'high_vibration': 'high_vibration',
        'high temperature': 'high_temperature',
        'high_temperature': 'high_temperature',
        'temp_high': 'high_temperature',
        'over current': 'overcurrent',
        'overcurrent': 'overcurrent',
        'unknown': 'unknown'
    }
    return mapping.get(v, v)

alarms_clean['alarm_type_clean'] = alarms_clean['alarm_type'].map(normalize_alarm).fillna('unknown')
alarms_clean = alarms_clean.drop_duplicates(subset=['machine', 'timestamp', 'alarm_type_clean'], keep='first').reset_index(drop=True)
alarms_clean['timestamp'] = pd.to_datetime(alarms_clean['timestamp'], errors='coerce')

# Convert metadata dates
machines_clean['installation_date'] = pd.to_datetime(machines_clean['installation_date'], errors='coerce')
maintenance_clean['service_date'] = pd.to_datetime(maintenance_clean['service_date'], errors='coerce')
maintenance_clean['downtime'] = pd.to_numeric(maintenance_clean['downtime'], errors='coerce')

# Sensor outlier flags using domain-aware checks and IQR
for col in ['vibration', 'temperature', 'current']:
    q1 = sensor_clean[col].quantile(0.25)
    q3 = sensor_clean[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    sensor_clean[f'{col}_outlier_flag'] = (sensor_clean[col] < low) | (sensor_clean[col] > high)
    sensor_clean[f'{col}_iqr_low'] = low
    sensor_clean[f'{col}_iqr_high'] = high

# Save cleaned datasets
for name, df in [('sensor_clean', sensor_clean), ('machines_clean', machines_clean), ('usage_clean', usage_clean), ('alarms_clean', alarms_clean), ('maintenance_clean', maintenance_clean)]:
    df.to_csv(CLEAN_DIR / f'{name}.csv', index=False)

print('Rows removed by duplicate key rule:', len(sensor) - len(sensor_clean))
print('Rows removed by exact duplicate rule (sensor):', sensor.duplicated().sum())
print('Missing sensor values after cleaning:')
print(sensor_clean[['vibration', 'temperature', 'current']].isnull().sum().to_string())
print('\nAlarm label mapping summary:')
print(alarms_clean['alarm_type_clean'].value_counts(dropna=False).head(15).to_string())


Rows removed by duplicate key rule: 1
Rows removed by exact duplicate rule (sensor): 1
Missing sensor values after cleaning:
vibration      12
temperature    12
current        11

Alarm label mapping summary:
alarm_type_clean
general_warning     636
high_vibration      218
high_temperature     64
overcurrent          26
unknown               4


## Data Integration and Merge Validation
The reusable grain is:
- sensor: one reading per machine per time point
- machines: one row per machine
- usage: one row per machine-week
- alarms: one event per machine-time-label after normalization
- maintenance: one maintenance event per service record

We validate row counts before and after each merge. This prevents accidental many-to-many joins and inflated denominators.


In [25]:
# Validate machine identifiers before merging
print('Unmatched sensor machines:', sensor_clean[~sensor_clean['machine'].isin(machines_clean['machine'])].shape[0])
print('Unmatched usage machines:', usage_clean[~usage_clean['machine'].isin(machines_clean['machine'])].shape[0])
print('Unmatched alarm machines:', alarms_clean[~alarms_clean['machine'].isin(machines_clean['machine'])].shape[0])
print('Unmatched maintenance machines:', maintenance_clean[~maintenance_clean['machine'].isin(machines_clean['machine'])].shape[0])

sensor_machine = sensor_clean.merge(machines_clean, on='machine', how='left', validate='many_to_one')
print('Sensor rows after merge with machine metadata:', len(sensor_machine))
print('Sensor machine join missing values:', sensor_machine['type'].isna().sum())

usage_week = usage_clean.copy()
usage_week['week_start'] = pd.to_datetime(usage_week['week_start'], errors='coerce')
usage_week = usage_week.sort_values(['machine', 'week_start']).reset_index(drop=True)

alarm_week = (
    alarms_clean.groupby(['machine', pd.Grouper(key='timestamp', freq='W-SUN')], as_index=False)
    .agg(alarm_count=('alarm_type_clean', 'size'), unique_alarm_types=('alarm_type_clean', 'nunique'))
    .rename(columns={'timestamp': 'week_start'})
)
usage_week = usage_week.merge(alarm_week, on=['machine', 'week_start'], how='left')
usage_week['alarm_count'] = usage_week['alarm_count'].fillna(0).astype(int)
usage_week['unique_alarm_types'] = usage_week['unique_alarm_types'].fillna(0).astype(int)

machine_sensor_summary = (
    sensor_clean.groupby('machine', as_index=False)
    .agg(
        sensor_reading_count=('timestamp', 'count'),
        mean_vibration=('vibration', 'mean'),
        max_vibration=('vibration', 'max'),
        mean_temperature=('temperature', 'mean'),
        max_temperature=('temperature', 'max'),
        mean_current=('current', 'mean'),
        max_current=('current', 'max'),
        vibration_outliers=('vibration_outlier_flag', 'sum'),
        temperature_outliers=('temperature_outlier_flag', 'sum'),
        current_outliers=('current_outlier_flag', 'sum')
    )
)

maintenance_summary = (
    maintenance_clean.groupby('machine', as_index=False)
    .agg(
        service_event_count=('service_date', 'size'),
        total_known_downtime=('downtime', 'sum'),
        mean_known_downtime=('downtime', 'mean'),
        first_service=('service_date', 'min'),
        last_service=('service_date', 'max')
    )
)
maintenance_summary['repeat_service_flag'] = (maintenance_summary['service_event_count'] >= 2).astype(int)

machine_level_summary = machines_clean.merge(machine_sensor_summary, on='machine', how='left', validate='one_to_one')
machine_level_summary = machine_level_summary.merge(usage_clean.groupby('machine', as_index=False).agg(mean_load=('load', 'mean'), mean_runtime=('runtime', 'mean'), total_cycles=('cycles', 'sum')), on='machine', how='left')
machine_level_summary = machine_level_summary.merge(maintenance_summary, on='machine', how='left', validate='one_to_one')
machine_level_summary['days_since_installation'] = (pd.Timestamp('today') - machine_level_summary['installation_date']).dt.days / 365.25
machine_level_summary['service_event_count'] = machine_level_summary['service_event_count'].fillna(0).astype(int)
machine_level_summary['total_known_downtime'] = machine_level_summary['total_known_downtime'].fillna(0)
machine_level_summary['repeat_service_flag'] = machine_level_summary['repeat_service_flag'].fillna(0).astype(int)

machine_level_summary.to_csv(OUT_DIR / 'machine_level_summary.csv', index=False)
print(machine_level_summary.head().to_string(index=False))


Unmatched sensor machines: 0
Unmatched usage machines: 0
Unmatched alarm machines: 0
Unmatched maintenance machines: 0
Sensor rows after merge with machine metadata: 31010
Sensor machine join missing values: 0
machine            type  age installation_date  sensor_reading_count  mean_vibration  max_vibration  mean_temperature  max_temperature  mean_current  max_current  vibration_outliers  temperature_outliers  current_outliers  mean_load  mean_runtime  total_cycles  service_event_count  total_known_downtime  mean_known_downtime first_service last_service  repeat_service_flag  days_since_installation
   M001            Pump  0.9        2025-11-12                   519        3.161060           5.61         70.405202             92.6     11.639440        17.16                   4                     6                 3  54.741026    114.715385         51951                    3                  26.7             8.900000    2026-06-24   2026-07-30                    1                 0.9

In [26]:
# EDA: numeric distributions and trends
sns.set_theme(style='whitegrid')

for col in ['vibration', 'temperature', 'current', 'runtime', 'load', 'cycles']:
    source = sensor_clean if col in sensor_clean.columns else usage_clean
    if col in source.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(source[col].dropna(), kde=True, ax=ax, bins=30)
        ax.set_title(f'Distribution of {col.title()}')
        ax.set_xlabel(col.title())
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f'{col}_distribution.png', dpi=150)
        plt.close(fig)

for col in ['vibration', 'temperature', 'current']:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(y=sensor_clean[col].dropna(), ax=ax, color='lightsteelblue')
    ax.set_title(f'{col.title()} Boxplot')
    ax.set_ylabel(col.title())
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f'{col}_boxplot.png', dpi=150)
    plt.close(fig)

for col in ['vibration', 'temperature', 'current']:
    temp = sensor_clean[['timestamp', 'machine', col]].dropna(subset=[col]).sort_values('timestamp')
    if not temp.empty:
        fig, ax = plt.subplots(figsize=(12, 4))
        for machine, g in temp.groupby('machine'):
            g = g.sort_values('timestamp')
            if machine in ['M001', 'M002', 'M003', 'M004', 'M005']:
                ax.plot(g['timestamp'], g[col], linewidth=0.8, alpha=0.4, label=machine)
        ax.set_title(f'{col.title()} Over Time')
        ax.set_xlabel('Timestamp')
        ax.set_ylabel(col.title())
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f'{col}_over_time.png', dpi=150)
        plt.close(fig)

alarm_frequency = alarms_clean['alarm_type_clean'].value_counts().rename_axis('alarm_type').reset_index(name='count')
alarm_frequency.to_csv(OUT_DIR / 'alarm_frequency.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=alarm_frequency.head(10), x='count', y='alarm_type', ax=ax, palette='viridis')
ax.set_title('Normalized Alarm Frequency')
ax.set_xlabel('Count')
ax.set_ylabel('Alarm Type')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'alarm_frequency.png', dpi=150)
plt.close(fig)

maintenance_action_counts = maintenance_clean['action'].fillna('unknown').value_counts().rename_axis('action').reset_index(name='count')
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=maintenance_action_counts.head(10), x='count', y='action', ax=ax, palette='magma')
ax.set_title('Maintenance Action Frequency')
ax.set_xlabel('Count')
ax.set_ylabel('Action')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'maintenance_actions.png', dpi=150)
plt.close(fig)

machine_type_downtime = machine_level_summary[['type', 'total_known_downtime']].dropna()
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=machine_type_downtime, x='type', y='total_known_downtime', ax=ax)
ax.set_title('Known Downtime by Machine Type')
ax.set_xlabel('Machine Type')
ax.set_ylabel('Known Downtime (hrs)')
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'downtime_by_type.png', dpi=150)
plt.close(fig)

num_cols = [c for c in machine_level_summary.select_dtypes(include=[np.number]).columns if c != 'age']
if len(num_cols) > 1:
    corr = machine_level_summary[num_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Machine-Level Feature Correlation Heatmap')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'feature_correlation.png', dpi=150)
    plt.close(fig)

print('EDA plots saved to:', PLOTS_DIR)


EDA plots saved to: c:\Users\Hari Vignesh S\Downloads\MechaCare_Final_Assessment_Package1\MechaCare_Final_Assessment_Package\plots


## Answers to Six Business Questions

The following sections answer each question with the actual cleaned data and explicit assumptions.


In [ ]:
# Q1. Normal and abnormal sensor ranges
q1_summary = sensor_clean[['vibration', 'temperature', 'current']].describe().T
q1_summary['iqr_low'] = [sensor_clean[col].quantile(0.25) - 1.5 * (sensor_clean[col].quantile(0.75) - sensor_clean[col].quantile(0.25)) for col in ['vibration', 'temperature', 'current']]
q1_summary['iqr_high'] = [sensor_clean[col].quantile(0.75) + 1.5 * (sensor_clean[col].quantile(0.75) - sensor_clean[col].quantile(0.25)) for col in ['vibration', 'temperature', 'current']]
q1_summary['outlier_count'] = [
    int(((sensor_clean[col] < q1_summary.loc[col, 'iqr_low']) | (sensor_clean[col] > q1_summary.loc[col, 'iqr_high'])).sum())
    for col in ['vibration', 'temperature', 'current']
]
print(q1_summary[['count', 'mean', 'min', '25%', '50%', '75%', 'max', 'iqr_low', 'iqr_high', 'outlier_count']].to_string())


In [27]:
# Q2. Sensor readings before maintenance
# Use the next maintenance event for each sensor record and machine.
pre_maintenance_records = []
for machine, g in sensor_clean.groupby('machine'):
    service_dates = maintenance_clean.loc[maintenance_clean['machine'] == machine, 'service_date'].sort_values().tolist()
    for _, row in g.sort_values('timestamp').iterrows():
        next_service = next((d for d in service_dates if pd.notna(d) and d > row['timestamp']), pd.NaT)
        if pd.isna(next_service):
            continue
        days_to_next = (next_service - row['timestamp']).days
        if 0 <= days_to_next <= 21:
            pre_maintenance_records.append({
                'machine': machine,
                'timestamp': row['timestamp'],
                'vibration': row['vibration'],
                'temperature': row['temperature'],
                'current': row['current'],
                'days_to_next_service': days_to_next,
                'next_service_date': next_service
            })

pre_maintenance = pd.DataFrame(pre_maintenance_records)
if not pre_maintenance.empty:
    pre_maintenance['window'] = pd.cut(
        pre_maintenance['days_to_next_service'],
        bins=[-1, 3, 7, 14, 21],
        labels=['0-3 days', '4-7 days', '8-14 days', '15-21 days'],
        include_lowest=True
    )
    pre_maintenance_summary = (
        pre_maintenance.groupby('window', observed=False)
        .agg(
            readings=('timestamp', 'count'),
            mean_vibration=('vibration', 'mean'),
            mean_temperature=('temperature', 'mean'),
            mean_current=('current', 'mean')
        )
        .reset_index()
    )
    pre_maintenance_summary.to_csv(OUT_DIR / 'pre_maintenance_sensor_summary.csv', index=False)
    print(pre_maintenance_summary.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=pre_maintenance_summary, x='window', y='mean_vibration', ax=ax, palette='viridis')
    ax.set_title('Mean Vibration by Days to Next Service')
    ax.set_xlabel('Days to Next Service')
    ax.set_ylabel('Mean Vibration')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'before_service_trend.png', dpi=150)
    plt.close(fig)
else:
    print('No readings within the 21-day pre-service window were found.')


    window  readings  mean_vibration  mean_temperature  mean_current
  0-3 days       691        3.833821         76.988567     13.445109
  4-7 days       766        3.836005         76.428068     13.284347
 8-14 days      1256        3.782062         75.906290     13.158804
15-21 days      1133        3.018074         68.976787     11.430635


In [28]:
# Q3. Which alarm types occur most frequently?
alarm_frequency = alarms_clean['alarm_type_clean'].value_counts().rename_axis('alarm_type').reset_index(name='count')
print(alarm_frequency.head(10).to_string(index=False))
alarm_frequency.to_csv(OUT_DIR / 'alarm_frequency.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=alarm_frequency.head(10), x='count', y='alarm_type', ax=ax, palette='Blues_d')
ax.set_title('Most Frequent Normalized Alarm Types')
ax.set_xlabel('Alarm Count')
ax.set_ylabel('Alarm Type')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'alarm_frequency.png', dpi=150)
plt.close(fig)


      alarm_type  count
 general_warning    636
  high_vibration    218
high_temperature     64
     overcurrent     26
         unknown      4


In [29]:
# Q4. Relationship between load and downtime
# Use machine-level aggregation; we do not attach machine downtime to every weekly row as if they were independent observations.
machine_usage = usage_clean.groupby('machine', as_index=False).agg(mean_load=('load', 'mean'), mean_runtime=('runtime', 'mean'), total_cycles=('cycles', 'sum'))
machine_downtime = maintenance_clean.groupby('machine', as_index=False).agg(total_known_downtime=('downtime', 'sum'))
load_downtime = machine_usage.merge(machine_downtime, on='machine', how='left')
load_downtime = load_downtime.dropna(subset=['total_known_downtime'])
print('Machine-level correlation (load vs downtime):', load_downtime[['mean_load', 'total_known_downtime']].corr().iloc[0, 1])

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=load_downtime, x='mean_load', y='total_known_downtime', hue='machine', palette='tab10', s=100, legend=False, ax=ax)
ax.set_title('Machine Load vs Total Known Downtime')
ax.set_xlabel('Average Load')
ax.set_ylabel('Total Known Downtime (hours)')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'load_vs_downtime.png', dpi=150)
plt.close(fig)

load_downtime.to_csv(OUT_DIR / 'load_downtime_analysis.csv', index=False)


Machine-level correlation (load vs downtime): 0.04088351196603776


In [30]:
# Q5. Which machine types require more servicing?
machine_type_summary = (
    machine_level_summary.groupby('type', as_index=False)
    .agg(
        machine_count=('machine', 'nunique'),
        total_service_events=('service_event_count', 'sum'),
        avg_service_events_per_machine=('service_event_count', 'mean'),
        total_known_downtime=('total_known_downtime', 'sum'),
        avg_known_downtime_per_machine=('total_known_downtime', 'mean'),
        repeat_service_machines=('repeat_service_flag', 'sum')
    )
    .sort_values('avg_service_events_per_machine', ascending=False)
    .reset_index(drop=True)
)
print(machine_type_summary.to_string(index=False))
machine_type_summary.to_csv(OUT_DIR / 'machine_type_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=machine_type_summary, x='type', y='avg_service_events_per_machine', ax=ax, palette='Set2')
ax.set_title('Average Service Events per Machine Type')
ax.set_xlabel('Machine Type')
ax.set_ylabel('Average Service Events per Machine')
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'machine_type_services.png', dpi=150)
plt.close(fig)


           type  machine_count  total_service_events  avg_service_events_per_machine  total_known_downtime  avg_known_downtime_per_machine  repeat_service_machines
     Compressor              7                    17                        2.428571                 133.8                       19.114286                        6
            CNC             15                    29                        1.933333                 294.5                       19.633333                        9
Hydraulic Press             11                    21                        1.909091                 178.7                       16.245455                        7
       Conveyor              9                    15                        1.666667                 111.5                       12.388889                        4
           Pump             18                    24                        1.333333                 190.3                       10.572222                        6


In [31]:
# Q6. Repeated failure patterns and machine-priority shortlist
priority = machine_level_summary.copy()
priority['alarm_count'] = alarms_clean.groupby('machine').size().reindex(priority['machine']).fillna(0).astype(int).to_numpy()
priority['warning_score'] = (
    (priority['max_vibration'] >= 5).astype(int) * 2 +
    (priority['max_temperature'] >= 85).astype(int) * 2 +
    (priority['max_current'] >= 16).astype(int) * 2 +
    priority['service_event_count'] * 2 +
    priority['repeat_service_flag'] * 3 +
    priority['alarm_count'] * 0.5
)
priority = priority.sort_values('warning_score', ascending=False).reset_index(drop=True)
print(priority[['machine','type','service_event_count','repeat_service_flag','max_vibration','max_temperature','max_current','alarm_count','warning_score']].head(10).to_string(index=False))
priority.to_csv(OUT_DIR / 'machine_priority_ranking.csv', index=False)


machine            type  service_event_count  repeat_service_flag  max_vibration  max_temperature  max_current  alarm_count  warning_score
   M053             CNC                    4                    1           7.28             99.2        20.47           68           51.0
   M004            Pump                    4                    1           6.80            107.7        19.83           63           48.5
   M042 Hydraulic Press                    4                    1           6.96             99.8        21.07           63           48.5
   M027 Hydraulic Press                    4                    1           6.77            104.1        19.21           56           45.0
   M011             CNC                    2                    1           6.52             98.7        19.37           37           31.5
   M019             CNC                    2                    1           7.40             95.4        18.15           29           27.5
   M038             CNC    

## Feature Engineering
The next section creates representative features that are useful for descriptive analysis and model-ready feature engineering. These are retrospective features and should not be mistaken for a validated predictive maintenance model.


In [ ]:
# Feature engineering for descriptive analysis
machine_features = machine_level_summary.copy()
machine_features['days_since_installation'] = (pd.Timestamp('today') - machine_features['installation_date']).dt.days / 365.25

usage_features = usage_clean.groupby('machine', as_index=False).agg(
    mean_load=('load', 'mean'),
    mean_runtime=('runtime', 'mean'),
    total_cycles=('cycles', 'sum')
)

alarm_counts = alarms_clean.groupby('machine', as_index=False).agg(alarm_count=('alarm_type_clean', 'size'))

maintenance_features = maintenance_clean.groupby('machine', as_index=False).agg(
    service_event_count=('service_date', 'size'),
    total_known_downtime=('downtime', 'sum'),
    repeat_service_flag=('service_date', lambda s: int((s.size >= 2)))
)

feature_frame = machines_clean.merge(
    machine_features[['machine', 'sensor_reading_count', 'mean_vibration', 'max_vibration', 'mean_temperature', 'max_temperature', 'mean_current', 'max_current', 'days_since_installation']],
    on='machine',
    how='left'
)
feature_frame = feature_frame.merge(usage_features, on='machine', how='left')
feature_frame = feature_frame.merge(alarm_counts, on='machine', how='left')
feature_frame = feature_frame.merge(maintenance_features, on='machine', how='left')
feature_frame['alarm_count'] = feature_frame['alarm_count'].fillna(0).astype(int)
feature_frame['service_event_count'] = feature_frame['service_event_count'].fillna(0).astype(int)
feature_frame['total_known_downtime'] = feature_frame['total_known_downtime'].fillna(0)
feature_frame['repeat_service_flag'] = feature_frame['repeat_service_flag'].fillna(0).astype(int)
print(feature_frame.head().to_string(index=False))


## Optional PCA Analysis
This section is intentionally optional and separate from the assessment conclusions. It standardizes numerical inputs and checks how much variance a small PCA model captures before any real predictive use.


In [33]:
# Optional PCA on machine-level numeric features
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

if 'feature_frame' not in globals():
    feature_frame = machine_level_summary.copy()
    feature_frame['days_since_installation'] = (pd.Timestamp('today') - feature_frame['installation_date']).dt.days / 365.25

feature_numeric = feature_frame.select_dtypes(include=[np.number]).drop(columns=['age'], errors='ignore')
feature_numeric = feature_numeric.dropna(axis=1, how='any')
if feature_numeric.shape[1] > 1:
    X = StandardScaler().fit_transform(feature_numeric)
    pca = PCA(n_components=min(3, X.shape[1]))
    X_pca = pca.fit_transform(X)
    print('PCA explained variance ratio:', pca.explained_variance_ratio_.round(3).tolist())
    print('Cumulative explained variance:', np.cumsum(pca.explained_variance_ratio_).round(3).tolist())
else:
    print('Not enough numeric features for PCA.')


PCA explained variance ratio: [0.428, 0.187, 0.086]
Cumulative explained variance: [0.428, 0.615, 0.7]


## Final Findings and Recommendations
- The data quality issues are manageable and should be documented clearly.
- Sensor readings should be checked by `(machine, timestamp)` rather than timestamp alone.
- Alarm labels must be normalized before counting.
- Pre-maintenance windows must be aligned to each machine's next service event, not all events.
- Machine-level aggregation is the defensible basis for load-versus-downtime analysis.
- The project should report uncertainty and avoid claiming causation from observed correlation alone.
